In [ ]:
# Add project root to Python path and change working directory
import sys
import os
from pathlib import Path

# Get project root
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))

# Change working directory to project root so config.yaml can be found
os.chdir(project_root)

print(f"Added to path: {project_root}")
print(f"Working directory: {os.getcwd()}")

In [3]:
from src.indexing.vector_store import get_vector_store
from src.indexing.embedder import get_embedder

store = get_vector_store()
embedder = get_embedder()

2026-02-16 14:57:30 - src.indexing.embedder - INFO - Loading sentence transformer model: all-mpnet-base-v2
2026-02-16 14:57:30 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2
2026-02-16 14:57:34 - src.indexing.embedder - INFO - Model loaded. Dimension: 768


In [ ]:
# Test retrieve_by_ids
chunk_ids = ["151caa59-5e0c-497e-9540-eae1de9229d4"]
results = store.retrieve_by_ids(chunk_ids)

for r in results:
    print(f"Chunk ID: {r.chunk_id}")
    print(f"Company: {r.metadata['company']}")
    print(f"Doc Type: {r.metadata['document_type']}")
    print(f"Page: {r.metadata['page_number']}")
    print(f"Content: {r.content}...")
    print()

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range
# Filter for page_number == 64
page_filter = Filter(
    must=[
        FieldCondition(key="page_number", range=Range(gte=64, lte=64)),
        FieldCondition(key="company", match=MatchValue(value="AAPL")) 
    ]
)
# Use scroll to get all matching points (no vector search)
results = store.client.scroll(
    collection_name="sec_filings",
    scroll_filter=page_filter,
    limit=100,  # Max results to return
    with_payload=True,
    with_vectors=False
)

for result in results:
    print(result)
    print("\n\n")

In [23]:
# Get unique values for the "company" field
unique_companies = store.client.facet(
    collection_name="sec_filings",
    key="company"
)
companies = [company.value for company in unique_companies.hits]
available_companies_str = ", ".join(companies)
print(available_companies_str)



2026-02-16 16:43:02 - httpx - INFO - HTTP Request: POST https://c0ab454e-b357-44c0-b7ac-8eb8a938d950.us-east-1-1.aws.cloud.qdrant.io:6333/collections/sec_filings/facet "HTTP/1.1 200 OK"
JPM, NVDA, MSFT, BRKB, META, TSLA, AVGO, AMZN, AAPL


In [26]:
QUESTION_GENERATION_SYSTEM = """You are a financial document analyst. Generate hypothetical questions that a financial document chunk could answer.

CRITICAL - Include temporal and company context in EVERY question:
- ALWAYS include the company ticker or name
- ALWAYS include the time period (fiscal year, quarter, or specific date)
- Include document type when relevant (10-K, 10-Q, 8-K)

Generate {questions_per_chunk} diverse questions that:
1. Cover different aspects or metrics in the chunk
2. Are specific and directly answerable from the chunk content
3. Use natural language a financial analyst would use
4. Include explicit temporal markers (e.g., "in fiscal year 2024", "for Q3 2024", "as of November 2024")

Return ONLY valid JSON:
{{
  "questions": [
    "question 1 with company and time period",
    "question 2 with company and time period",
    "question 3 with company and time period"
  ]
}}
"""

q = QUESTION_GENERATION_SYSTEM.format(questions_per_chunk=1)
print(q)

You are a financial document analyst. Generate hypothetical questions that a financial document chunk could answer.

CRITICAL - Include temporal and company context in EVERY question:
- ALWAYS include the company ticker or name
- ALWAYS include the time period (fiscal year, quarter, or specific date)
- Include document type when relevant (10-K, 10-Q, 8-K)

Generate 1 diverse questions that:
1. Cover different aspects or metrics in the chunk
2. Are specific and directly answerable from the chunk content
3. Use natural language a financial analyst would use
4. Include explicit temporal markers (e.g., "in fiscal year 2024", "for Q3 2024", "as of November 2024")

Return ONLY valid JSON:
{
  "questions": [
    "question 1 with company and time period",
    "question 2 with company and time period",
    "question 3 with company and time period"
  ]
}

